In [1]:
# Cell 1 - Imports
import requests
import pandas as pd
import time
import warnings
warnings.filterwarnings('ignore')

In [2]:
# Cell 2 - Load existing data (1920-2021)
df = pd.read_csv('merged_article_counts_by_country_across_years.csv')
print(f"Loaded: {len(df)} countries, years 1920-2021")


Loaded: 197 countries, years 1920-2021


In [3]:
# Cell 3 - Fetch article counts for 2022-2025 & update totals to 1920-2025
api_key = 'eb2cec89a26c7449245d9379a4f9944e'
base_url = 'https://api.elsevier.com/content/search/scopus'
headers = {'Accept': 'application/json'}

# 6 countries where Scopus uses a different name than the backbone
scopus_name_map = {
    'United States of America': 'United States',
    'Hong Kong S.A.R.': 'Hong Kong',
    'Vietnam': 'Viet Nam',
    'Syria': 'Syrian Arab Republic',
    'Ivory Coast': "Cote d'Ivoire",
    'Democratic Republic of the Congo': 'Democratic Republic Congo',
}

for i in range(len(df)):
    country = df.loc[i, 'country']
    if pd.isnull(country):
        continue

    # Use Scopus-compatible name (if necessary) 
    query_country = scopus_name_map.get(country, country)

    print(f"[{i+1}/{len(df)}] {country}...", end=" ", flush=True)

    for year in range(2022, 2026):
        query = f'TITLE-ABS-KEY("anesthes*" OR "anaesthes*") AND AFFILCOUNTRY({query_country})'
        params = {'query': query, 'count': 25, 'date': str(year), 'apiKey': api_key}
        response = requests.get(base_url, headers=headers, params=params)
        if response.status_code == 200:
            data = response.json()
            total = data.get('search-results', {}).get('opensearch:totalResults', 0)
            df.loc[i, str(year)] = int(total)
        time.sleep(1)

    # Update total count (1920-2025)
    query = f'TITLE-ABS-KEY("anesthes*" OR "anaesthes*") AND AFFILCOUNTRY({query_country})'
    params = {'query': query, 'count': 25, 'date': '1920-2025', 'apiKey': api_key}
    response = requests.get(base_url, headers=headers, params=params)
    if response.status_code == 200:
        data = response.json()
        total = data.get('search-results', {}).get('opensearch:totalResults', 0)
        df.loc[i, 'anesthesiology'] = int(total)

    print("done.")

[1/197] Afghanistan... done.
[2/197] Albania... done.
[3/197] Algeria... done.
[4/197] Andorra... done.
[5/197] Angola... done.
[6/197] Antigua and Barbuda... done.
[7/197] Argentina... done.
[8/197] Armenia... done.
[9/197] Australia... done.
[10/197] Austria... done.
[11/197] Azerbaijan... done.
[12/197] Bahamas... done.
[13/197] Bahrain... done.
[14/197] Bangladesh... done.
[15/197] Barbados... done.
[16/197] Belarus... done.
[17/197] Belgium... done.
[18/197] Belize... done.
[19/197] Benin... done.
[20/197] Bhutan... done.
[21/197] Bolivia... done.
[22/197] Bosnia and Herzegovina... done.
[23/197] Botswana... done.
[24/197] Brazil... done.
[25/197] Brunei... done.
[26/197] Bulgaria... done.
[27/197] Burkina Faso... done.
[28/197] Burundi... done.
[29/197] Cabo Verde... done.
[30/197] Cambodia... done.
[31/197] Cameroon... done.
[32/197] Canada... done.
[33/197] Central African Republic... done.
[34/197] Chad... done.
[35/197] Chile... done.
[36/197] China... done.
[37/197] Colombia

In [4]:
# Cell 4 - Update HDI to 2023 values from 2025 HDR
hdi_new = pd.read_csv('backbone_anesthesiology_2023hdi.csv')
hdi_lookup = hdi_new.set_index('country')[['hdi_2023', 'hdicode']].to_dict('index')

for i, row in df.iterrows():
    if row['country'] in hdi_lookup:
        info = hdi_lookup[row['country']]
        df.loc[i, 'hdi_2023'] = info['hdi_2023']
        df.loc[i, 'hdicode'] = info['hdicode']

if 'hdi_2022' in df.columns:
    df = df.drop(columns=['hdi_2022'])

In [5]:
# Cell 5 - Save
df.to_csv('UPDATED_merged_article_counts_by_country_across_years.csv', index=False)
print(f"Saved! Shape: {df.shape}")
print(df[['country', 'hdicode', 'hdi_2023', 'anesthesiology', '2022', '2023', '2024', '2025']].head(10))



Saved! Shape: (197, 112)
               country    hdicode  hdi_2023  anesthesiology   2022   2023  \
0          Afghanistan        Low     0.496              49    4.0    7.0   
1              Albania  Very High     0.810              63    5.0    5.0   
2              Algeria       High     0.763             103    2.0   11.0   
3              Andorra  Very High     0.913               4    0.0    1.0   
4               Angola     Medium     0.616              11    0.0    0.0   
5  Antigua and Barbuda  Very High     0.851              10    1.0    1.0   
6            Argentina  Very High     0.865            1289   86.0   74.0   
7              Armenia  Very High     0.811              65    9.0   10.0   
8            Australia  Very High     0.958           13559  640.0  539.0   
9              Austria  Very High     0.930            4832  234.0  211.0   

    2024   2025  
0    3.0    9.0  
1   11.0   10.0  
2   19.0   18.0  
3    1.0    0.0  
4    1.0    3.0  
5    2.0    1.0  
6